# 100 — Búsqueda híbrida y fusión de rankings

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Búsqueda híbrida**: ejecutar en paralelo BM25 (literal, clase 099) y denso
(semántico, clase 097) y fusionar. Problema: los scores de motores distintos son
incomparables (BM25 no acotado vs coseno en [−1,1]).

**Reciprocal Rank Fusion** (Cormack et al., SIGIR 2009) fusiona usando solo posiciones:

```text
RRF(d) = Σ_r 1/(k + rank_r(d)),   k ≈ 60
```

- El recíproco concentra crédito en los primeros puestos.
- `k` amortigua: con k=60 el puesto 1 vale 1/61, el 10 vale 1/70 — el consenso entre
  rankings pesa más que un primer puesto aislado.
- Inmune a escalas y calibración: solo necesita listas ordenadas.

Variantes: RRF ponderado (`w_r/(k+rank)`), score fusion con min-max + combinación
convexa (frágil: exige calibrar α por colección).


## 🧮 Ejemplo de referencia

BM25 = [A, B, C, D]; Denso = [C, A, E, B]; k = 60:

```text
RRF(A) = 1/61 + 1/62 = 0.03252   ← gana sin ser 1.º en ninguno (consenso)
RRF(C) = 1/63 + 1/61 = 0.03227
RRF(B) = 1/62 + 1/64 = 0.03175
RRF(E) = 1/63 = 0.01587    RRF(D) = 1/64 = 0.01563   (una sola rama)
```

Fusión: A > C > B > E > D. Reproduce estas sumas a mano antes de ejecutar el laboratorio.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("retrieval", seed=100)
show(result)


## Reflexión

1. En el ejemplo, A gana sin ser primero en ningún ranking. ¿Qué propiedad de RRF produce eso y por qué suele ser deseable en recuperación?
2. ¿Qué pasaría con RRF si usaras k = 0? Calcula RRF(C) y RRF(A) del ejemplo con k = 0 y comprueba si el ganador cambia y por qué.
3. ¿En qué situación una fusión híbrida rendiría peor que la mejor rama sola, y qué experimento lo detectaría antes de desplegarla?
